In [8]:
import pandas as pd
import re

In [12]:
df = pd.read_csv(r"C:\Users\mdumiseni\Documents\data science assignements\data-science-portfolio\nutrition-risk-african-recipes\data_clean\recipe_ingredients.csv")
col = "ingredient_line_raw"

In [13]:
unit_patterns = {
    r"\btsp?s?\b": "teaspoon",
    r"\btbsp?s?\b": "tablespoon",
    r"\btbs\b": "tablespoon"
}
df["std_unit"]= df[col]

for pat, repl in unit_patterns.items():
    df["std_unit"] = df["std_unit"].str.replace(pat, repl, regex = True, flags = re.IGNORECASE)

df[["ingredient_line_raw","std_unit"]].head()    

,ingredient_line_raw,std_unit
0,2 tbsp (30 ml) olive oil,2 tablespoon (30 ml) olive oil
1,1 cup (211 g) basmati rice,1 cup (211 g) basmati rice
2,2 cups (480 ml) vegetable stock,2 cups (480 ml) vegetable stock
3,1 cup (201 g) brown lentils,1 cup (201 g) brown lentils
4,"2 cups (480 ml) water, plus more as needed","2 cups (480 ml) water, plus more as needed"


In [15]:
df["actual_measure"] = df["std_unit"].str.extract(r"\(([^)]*)\)", expand=False)
df[["std_unit", "actual_measure"]].head()

,std_unit,actual_measure
0,2 tablespoon (30 ml) olive oil,30 ml
1,1 cup (211 g) basmati rice,211 g
2,2 cups (480 ml) vegetable stock,480 ml
3,1 cup (201 g) brown lentils,201 g
4,"2 cups (480 ml) water, plus more as needed",480 ml


In [16]:
df["standard_unit"] = (
    df["std_unit"]
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.strip()
)

df[["std_unit", "actual_measure", "standard_unit"]].head()

,std_unit,actual_measure,standard_unit
0,2 tablespoon (30 ml) olive oil,30 ml,2 tablespoon olive oil
1,1 cup (211 g) basmati rice,211 g,1 cup basmati rice
2,2 cups (480 ml) vegetable stock,480 ml,2 cups vegetable stock
3,1 cup (201 g) brown lentils,201 g,1 cup brown lentils
4,"2 cups (480 ml) water, plus more as needed",480 ml,"2 cups water, plus more as needed"


In [23]:
units = [
    "cup", "cups",
    "teaspoon", "teaspoons", "tsp", "tsps",
    "tablespoon", "tablespoons", "tbsp", "tbsps",
    "clove", "cloves",
    "can", "cans", "small",
    "medium", "large", "cube"
]

In [27]:
import re

units_pattern = r"cup|cups|teaspoon|teaspoons|tsp|tsps|tablespoon|tablespoons|tbsp|tbsps|clove|cloves|can|cans|small|medium|large|cube"
qty_pattern = r"\d+(?:\.\d+)?|\d+\s*/\s*\d+|\d+\s+\d+/\d+|[¼½¾]"
pattern = rf"(?i)^\s*({qty_pattern})\s*({units_pattern})\b"

In [28]:
extracted = df["standard_unit"].str.extract(pattern, expand=True)
df["measure_only"] = (
    extracted[0].str.replace(r"\s+", " ", regex=True).str.strip()
    + " "
    + extracted[1].str.lower().str.strip()
)
df.loc[extracted[0].isna() | extracted[1].isna(), "measure_only"] = pd.NA

In [29]:
df["measure_only"].head()

0    None
1    None
2    None
3    None
4    None
Name: measure_only, dtype: object

In [22]:
df.to_csv(r"C:\Users\mdumiseni\Documents\data science assignements\data-science-portfolio\nutrition-risk-african-recipes\data_clean\recipe_ingredients_cleaned.csv", index = False)